**Question:** Do the rankings and the percentage gains over ELO of notebooks 14-17 survive when every model is compared on equal terms: ELO recalibrated like the other models, raw and calibrated outputs for every model, the same number of seeds for the GRU and XGBoost ensembles, and symmetrised versions side by side?

**Assumptions:** Same test matches and folds as notebooks 14-17 (walk-forward folds 1-4, 14,428 matches pooled, 28,856 rows). The calibration slice is the chronologically last 15 % of each fold's train matches for every model. Blend weights are fixed at 0.5 / 0.5 in advance; nothing is chosen on test data.

**Method:** ELO is recalibrated per fold (isotonic, alternatively logistic on `elo_diff`) on the calibration slice. XGBoost B' and B'-par are re-trained with 5 seeds (`scripts/run_walkforward_xgb_seeds.py`), saving raw and calibrated probabilities. Every model is reported raw, calibrated and symmetrised (`p_sym = (p_AB + 1 - p_BA) / 2`). Paired bootstrap by match (2,000 resamples, seed 42, the same indices for every comparison), pooled over 4 folds; Brier, accuracy (both perspectives, a tie counts half a point) and AUC; percentage gain against both the raw and the recalibrated ELO.

**Result:** Recalibrating ELO lowers its Brier by 0.0045 (2.5 %), so every "% vs ELO" figure shrinks by about 2.3-2.6 percentage points when the reference is the calibrated ELO. The order of the models does not change (C3ens, C3, TabPFN-3.5, TabPFN v2, XGBoost B', XGBoost B'-par, ELO); calibration slightly hurts every non-ELO model, and with 5 seeds XGBoost B' is no longer statistically separable from TabPFN v2.

# 18 -- Fair comparison of all models: raw vs calibrated, seed parity, symmetrised versions

**Why.** Notebooks 14-17 compare models that were calibrated differently (ELO not at all, the GRU and TabPFN with a calibrator chosen on the calibration slice, XGBoost isotonic) and that were not equally seeded (the GRU ensemble has 5 seeds, XGBoost had 1). Percentages "vs ELO" therefore mix a modelling gain with a calibration gain. This notebook puts every model on the same footing and reports both versions of everything.

**What is fixed (same for every model).**
- Test data: walk-forward folds 1-4, 14,428 matches pooled (28,856 rows, both perspectives), by-match paired bootstrap, 2,000 resamples, seed 42, the same indices for every number.
- Calibration slice: the chronologically last 15 % of each fold's train matches (`carve_train` rule). ELO is recalibrated on exactly this slice (isotonic, and alternatively a logistic fit on `elo_diff`), then applied to the fold's test rows.
- Every model is shown **raw and calibrated**, with no choice made on test data. XGBoost B' and B'-par use 5 seeds (`random_state` 42, 1, 2, 3, 4) with the same hyper-parameters; the ensemble is the mean of the seed probabilities (raw with raw, calibrated with calibrated), exactly like the 5-seed GRU ensemble.
- Every model is also shown **symmetrised**, `p_sym = (p_AB + 1 - p_BA) / 2`. ELO and TabPFN are symmetric by construction and do not change.
- Blend weights are fixed at 0.5 / 0.5 in advance.

**Statistics.** Accuracy counts both perspectives, `p > 0.5`, an exact tie is half a point. `% vs ELO raw` / `% vs ELO cal` is `100 * (1 - Brier / Brier_ELO)` against the uncalibrated and against the isotonic-recalibrated ELO. Negative paired dBrier = first model better. No predictions are stored in this notebook.

## Step 0 -- Imports and data

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, '../scripts')
from exp_d_analysis import paired_dbrier
from fair_comparison import (load_fair, add_sym, add_fair_blends, fair_table, fold_brier, folds_better,
                             pool_folds, boot_idx, brier_ci, XGB_SEEDS)

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', None)

WORKDIR = os.environ.get('GDSI_ML_WORKDIR', '..')
# 5-seed XGBoost predictions (raw + calibrated per seed) are produced once by the training script (skipped if present)
for variant, stem in (('Bprime', 'bprime'), ('Bprime_par', 'bprimepar')):
    if not (Path(WORKDIR) / 'exp_fair' / f'test_predictions_xgbseeds_{stem}_fold4.parquet').exists():
        subprocess.run([sys.executable, '../scripts/run_walkforward_xgb_seeds.py', WORKDIR, '--variant', variant], check=True)

B = load_fair(WORKDIR)
BASE = ['p_ELO', 'p_ELOcal', 'p_ELOplatt', 'p_C3raw', 'p_C3', 'p_C3ens_raw', 'p_C3ens', 'p_v2raw', 'p_v2cal', 'p_v35raw', 'p_v35cal',
        'p_xgb_raw', 'p_xgb_cal', 'p_xgbpar_raw', 'p_xgbpar_cal', 'p_xgb_raw_s42', 'p_xgb_cal_s42', 'p_xgbpar_raw_s42', 'p_xgbpar_cal_s42']
add_sym(B, BASE)
BL = add_fair_blends(B)
pool = pool_folds(B)
yp = pool.Label.values
boots = boot_idx(pool.SusretId.values)
folds = sorted(B)
print('folds:', folds, '| pooled matches:', pool.SusretId.nunique(), '| rows:', len(pool))

## Step 1 -- Calibrated ELO and XGBoost seed parity

**ELO.** The raw ELO probability `1 / (1 + 10^(-elo_diff / 400))` is recalibrated on the fold's calibration slice, with an isotonic fit (main) and with a logistic fit on `elo_diff` (Platt, alternative). Both are fitted on the same rows as every other model's calibrator and never see the test rows.

**XGBoost seeds.** B' and B'-par are trained for seeds 42, 1, 2, 3, 4 (identical hyper-parameters, features, folds, early stopping and calibration slice). Seed 42 reproduces the earlier single-seed predictions exactly; the ensemble is the plain mean over seeds.

In [ ]:
rows = [{'fold': k, 'n_matches': B[k].SusretId.nunique(), 'ELO raw': fold_brier(B, 'p_ELO')[i], 'ELO iso': fold_brier(B, 'p_ELOcal')[i],
         'ELO Platt': fold_brier(B, 'p_ELOplatt')[i]} for i, k in enumerate(folds)]
print(pd.DataFrame(rows).round(5).to_string(index=False))
print()
for c, nm in (('p_ELO', 'ELO raw'), ('p_ELOcal', 'ELO isotonic'), ('p_ELOplatt', 'ELO Platt')):
    b, lo, hi = brier_ci(pool[c].values, yp, boots)
    print(f'{nm:14s} pooled Brier {np.mean((pool[c].values - yp) ** 2):.5f} [{lo:.5f}, {hi:.5f}]')
for a, b_ in (('p_ELOcal', 'p_ELO'), ('p_ELOplatt', 'p_ELO'), ('p_ELOplatt', 'p_ELOcal')):
    d, lo, hi = paired_dbrier(pool[a].values, pool[b_].values, yp, boots)
    print(f'{a:12s} vs {b_:10s} dBrier {d:+.5f} [{lo:+.5f}, {hi:+.5f}]  folds better {folds_better(B, a, b_)}/4')

In [ ]:
# XGBoost: single seed vs 5-seed ensemble, raw and calibrated (pooled Brier, spread over the seeds)
sd = {}
for stem, nm in (('bprime', "XGB B'"), ('bprimepar', "XGB B'-par")):
    parts = [pd.read_parquet(Path(WORKDIR) / 'exp_fair' / f'test_predictions_xgbseeds_{stem}_fold{k}.parquet') for k in folds]
    sd[nm] = {v: [np.mean(np.concatenate([(p[f'p_{v}_s{s}'] - p.Label).values ** 2 for p in parts])) for s in XGB_SEEDS] for v in ('raw', 'cal')}
rows = []
for nm, tag in (("XGB B'", 'xgb'), ("XGB B'-par", 'xgbpar')):
    for v in ('raw', 'cal'):
        rows.append({'model': nm, 'variant': v, 'seed 42': np.mean((pool[f'p_{tag}_{v}_s42'] - yp) ** 2),
                     'seed min': min(sd[nm][v]), 'seed max': max(sd[nm][v]), 'seed mean': np.mean(sd[nm][v]),
                     '5-seed ensemble': np.mean((pool[f'p_{tag}_{v}'] - yp) ** 2)})
print(pd.DataFrame(rows).round(5).to_string(index=False))
print()
for tag, nm in (('xgb', "XGB B'"), ('xgbpar', "XGB B'-par")):
    for v in ('raw', 'cal'):
        d, lo, hi = paired_dbrier(pool[f'p_{tag}_{v}'].values, pool[f'p_{tag}_{v}_s42'].values, yp, boots)
        print(f'{nm:11s} {v}: 5-seed ensemble vs seed 42 dBrier {d:+.5f} [{lo:+.5f}, {hi:+.5f}]  folds better {folds_better(B, f"p_{tag}_{v}", f"p_{tag}_{v}_s42")}/4')

TabPFN is not seed-matched here: it has an internal ensemble (`n_estimators=8`) and one run per fold (seed 42); on fold 4 TabPFN v2 additionally has seeds 1 and 2 (notebook 14), which are not used in this notebook.

## Step 2 -- Main table: every model, raw and calibrated, unsymmetrised and symmetrised

`Variant`: `raw` = model output as is; `cal` = the calibrator of that model's own pipeline, fitted on the fold's calibration slice (ELO: isotonic or Platt as labelled; GRU C3 / C3ens: temperature scaling or isotonic, whichever scores better on the calibration slice; TabPFN: isotonic; XGBoost: isotonic). `Sym` = `p_sym = (p_AB + 1 - p_BA) / 2`; `built in` = symmetric by construction. The ensemble rows (C3ens, XGBoost) are 5-seed means. Rows with TabPFN-3.5 are research-only.

In [ ]:
S = [('ELO', 'raw', 'built in', 'p_ELO'), ('ELO', 'cal (isotonic)', 'no', 'p_ELOcal'), ('ELO', 'cal (isotonic)', 'yes', 'p_ELOcal_sym'),
     ('ELO', 'cal (Platt)', 'no', 'p_ELOplatt'), ('ELO', 'cal (Platt)', 'yes', 'p_ELOplatt_sym')]
for nm, rawc, calc in (('C3 (1 seed)', 'p_C3raw', 'p_C3'), ('C3ens (5 seeds)', 'p_C3ens_raw', 'p_C3ens'),
                       ("XGBoost B' (5 seeds)", 'p_xgb_raw', 'p_xgb_cal'), ("XGBoost B'-par (5 seeds)", 'p_xgbpar_raw', 'p_xgbpar_cal')):
    for v, c in (('raw', rawc), ('cal', calc)):
        S += [(nm, v, 'no', c), (nm, v, 'yes', c + '_sym')]
for nm, rawc, calc in (('TabPFN v2 (10k)', 'p_v2raw', 'p_v2cal'), ('TabPFN-3.5 (full, research-only)', 'p_v35raw', 'p_v35cal')):
    S += [(nm, 'raw', 'built in', rawc), (nm, 'cal', 'built in', calc)]
for c, l in BL.items():
    S.append((l.replace(' / sym', ''), '-', 'yes' if c.endswith('_sym') else 'no', c))
MAIN = fair_table(pool, S, boots)
fmt = lambda r: (f"| {r['Model']} | {r['Variant']} | {r['Sym']} | {r['Brier']:.5f} [{r['lo']:.5f}, {r['hi']:.5f}] | {r['% vs ELO raw']:.1f}% | {r['% vs ELO cal']:.1f}% | "
                 f"{r['Acc %']:.2f}% [{r['Acc lo']:.2f}, {r['Acc hi']:.2f}] | {r['AUC']:.4f} |")
print('| Model | Variant | Sym | Brier [95% CI] | % vs ELO raw | % vs ELO cal | Accuracy [95% CI] | AUC |')
print('|---|---|---|---|---|---|---|---|')
for _, r in MAIN.iterrows():
    print(fmt(r))

## Step 3 -- Does calibration help or hurt? (paired, per model)

Paired dBrier of the calibrated version against the raw version of the same model (negative = calibration helps), pooled over the folds, with the number of folds (out of 4) in which the calibrated version is better.

In [ ]:
PAIRS = [('ELO (isotonic)', 'p_ELOcal', 'p_ELO'), ('ELO (Platt)', 'p_ELOplatt', 'p_ELO'), ('C3', 'p_C3', 'p_C3raw'), ('C3ens', 'p_C3ens', 'p_C3ens_raw'),
         ('TabPFN v2', 'p_v2cal', 'p_v2raw'), ('TabPFN-3.5', 'p_v35cal', 'p_v35raw'), ("XGB B'", 'p_xgb_cal', 'p_xgb_raw'), ("XGB B'-par", 'p_xgbpar_cal', 'p_xgbpar_raw')]
rows = []
for nm, a, b_ in PAIRS:
    d, lo, hi = paired_dbrier(pool[a].values, pool[b_].values, yp, boots)
    rows.append({'model': nm, 'Brier raw': np.mean((pool[b_].values - yp) ** 2), 'Brier cal': np.mean((pool[a].values - yp) ** 2),
                 'dBrier cal-raw': d, 'CI lo': lo, 'CI hi': hi, 'folds cal better': folds_better(B, a, b_)})
print(pd.DataFrame(rows).round(5).to_string(index=False))

## Step 4 -- Ranking: does the order of the models change?

Pooled Brier ranks of the single models: raw versions, calibrated versions, and the symmetrised versions of both (ELO and TabPFN are unchanged by symmetrisation). ELO is the uncalibrated reference in the raw ranking and the isotonic-recalibrated one in the calibrated ranking.

In [ ]:
def rank_of(cols):
    s = pd.Series({n: np.mean((pool[c].values - yp) ** 2) for n, c in cols.items()}).sort_values()
    return [f'{i + 1}. {n} ({v:.5f})' for i, (n, v) in enumerate(s.items())]
RAW = {'ELO': 'p_ELO', 'C3': 'p_C3raw', 'C3ens': 'p_C3ens_raw', 'TabPFN v2': 'p_v2raw', 'TabPFN-3.5': 'p_v35raw', "XGB B'": 'p_xgb_raw', "XGB B'-par": 'p_xgbpar_raw'}
CAL = {'ELO': 'p_ELOcal', 'C3': 'p_C3', 'C3ens': 'p_C3ens', 'TabPFN v2': 'p_v2cal', 'TabPFN-3.5': 'p_v35cal', "XGB B'": 'p_xgb_cal', "XGB B'-par": 'p_xgbpar_cal'}
RAWS = {n: (c if n.startswith(('ELO', 'TabPFN')) else c + '_sym') for n, c in RAW.items()}
CALS = {n: (c if n.startswith('TabPFN') else c + '_sym') for n, c in CAL.items()}
R = pd.DataFrame({'raw': rank_of(RAW), 'cal': rank_of(CAL), 'raw, sym': rank_of(RAWS), 'cal, sym': rank_of(CALS)})
print(R.to_string(index=False))

Paired dBrier between neighbouring single models (raw versions unless stated; negative = first model better), to see which of these orderings are statistically supported.

In [ ]:
CMP = [('p_C3ens_raw', 'p_C3raw'), ('p_C3ens_raw', 'p_v35raw'), ('p_C3raw', 'p_v35raw'), ('p_v35raw', 'p_v2raw'), ('p_v2raw', 'p_C3raw'),
       ('p_xgb_raw', 'p_v2raw'), ('p_xgb_cal', 'p_v2cal'), ('p_xgb_raw', 'p_C3raw'), ('p_xgbpar_raw', 'p_xgb_raw'), ('p_C3raw', 'p_ELOcal'), ('p_xgb_raw', 'p_ELOcal')]
LB = {'p_C3ens_raw': 'C3ens raw', 'p_C3raw': 'C3 raw', 'p_v35raw': 'TabPFN-3.5 raw', 'p_v2raw': 'TabPFN v2 raw', 'p_xgb_raw': "XGB B' raw (5 seeds)", 'p_xgb_cal': "XGB B' cal (5 seeds)",
      'p_v2cal': 'TabPFN v2 cal', 'p_xgbpar_raw': "XGB B'-par raw (5 seeds)", 'p_ELOcal': 'ELO isotonic'}
for a, b_ in CMP:
    d, lo, hi = paired_dbrier(pool[a].values, pool[b_].values, yp, boots)
    print(f'{LB[a]:26s} vs {LB[b_]:26s} dBrier {d:+.5f} [{lo:+.5f}, {hi:+.5f}]  folds better {folds_better(B, a, b_)}/4')

## Step 5 -- Blends with fixed 0.5 / 0.5 weights

The GRU part is the raw 5-seed ensemble (`C3ens raw`); the XGBoost part is the 5-seed ensemble, raw or calibrated, B' or B'-par, before and after symmetrisation of the blend. Paired dBrier against the raw 5-seed GRU ensemble and against the single raw GRU (`C3 raw`); `folds better` counts the folds (out of 4) with a lower Brier than the reference. The second block repeats the comparison of the symmetrised blends against the symmetrised raw GRUs. The rows with `C3ens cal + XGB cal` are the all-calibrated reference.

In [ ]:
CAND = [c for c in BL if c.startswith('b_C3ensraw_') and not c.endswith('_sym')]
CAND = [x for c in CAND for x in (c, c + '_sym')]
CAND += ['b_C3enscal_xgb_cal', 'b_C3enscal_xgb_cal_sym', 'b_C3enscal_xgbpar_cal', 'b_C3enscal_xgbpar_cal_sym']
def block(cands, refs):
    rows = []
    for c in cands:
        r = {'blend': BL[c], 'Brier': np.mean((pool[c].values - yp) ** 2)}
        for ref, nm in refs:
            d, lo, hi = paired_dbrier(pool[c].values, pool[ref].values, yp, boots)
            r[f'd vs {nm}'] = f'{d:+.5f} [{lo:+.5f}, {hi:+.5f}]'
            r[f'folds>{nm}'] = f'{folds_better(B, c, ref)}/4'
        rows.append(r)
    return pd.DataFrame(rows)
T = block(CAND, [('p_C3ens_raw', 'C3ens raw'), ('p_C3raw', 'C3 raw')])
print('| Blend | Brier | dBrier vs C3ens raw [95% CI] | folds better | dBrier vs C3 raw [95% CI] | folds better |')
print('|---|---|---|---|---|---|')
for _, r in T.iterrows():
    print(f"| {r['blend']} | {r['Brier']:.5f} | {r['d vs C3ens raw']} | {r['folds>C3ens raw']} | {r['d vs C3 raw']} | {r['folds>C3 raw']} |")
print()
print('Symmetrised blends against the symmetrised raw GRUs:')
T2 = block([c for c in CAND if c.endswith('_sym')], [('p_C3ens_raw_sym', 'C3ens raw sym'), ('p_C3raw_sym', 'C3 raw sym')])
print('| Blend | Brier | dBrier vs C3ens raw sym [95% CI] | folds better | dBrier vs C3 raw sym [95% CI] | folds better |')
print('|---|---|---|---|---|---|')
for _, r in T2.iterrows():
    print(f"| {r['blend']} | {r['Brier']:.5f} | {r['d vs C3ens raw sym']} | {r['folds>C3ens raw sym']} | {r['d vs C3 raw sym']} | {r['folds>C3 raw sym']} |")

## Step 6 -- Brier by fold

In [ ]:
cols = ['p_ELO', 'p_ELOcal', 'p_C3raw', 'p_C3ens_raw', 'p_v2raw', 'p_v35raw', 'p_xgb_raw', 'p_xgbpar_raw', 'b_C3ensraw_xgb_raw', 'b_C3ensraw_xgb_cal', 'b_C3ensraw_xgbpar_raw', 'b_C3ensraw_xgbpar_cal']
names = {'p_ELO': 'ELO raw', 'p_ELOcal': 'ELO iso', 'p_C3raw': 'C3 raw', 'p_C3ens_raw': 'C3ens raw', 'p_v2raw': 'TabPFN v2 raw', 'p_v35raw': 'TabPFN-3.5 raw',
         'p_xgb_raw': "XGB B' raw", 'p_xgbpar_raw': "XGB B'-par raw", 'b_C3ensraw_xgb_raw': "C3ens+XGB B' raw", 'b_C3ensraw_xgb_cal': "C3ens+XGB B' cal",
         'b_C3ensraw_xgbpar_raw': "C3ens+XGB B'-par raw", 'b_C3ensraw_xgbpar_cal': "C3ens+XGB B'-par cal"}
FT = pd.DataFrame({f'fold {k}': {names[c]: fold_brier(B, c)[i] for c in cols} for i, k in enumerate(folds)})
print(FT.round(5).to_string())

## Step 7 -- Full blend grid (fair conditions)

Every blend under the same conditions as the rest of this notebook: each component in its **raw** variant (raw is better for every learned model, Step 3), XGBoost as the 5-seed ensemble, GRU / XGBoost components symmetrised (`p_sym = (p_AB + 1 - p_BA) / 2`; TabPFN is symmetric by construction), weights fixed in advance (0.5 / 0.5 for 2-way, 1/3 each for 3-way; nothing fitted on test data). Rows are sorted by pooled Brier.

- GRU part: C3 raw (1 seed) or C3ens raw (5 seeds). Tabular part: XGB B' (5 seeds), XGB B'-par (5 seeds), TabPFN v2 raw, TabPFN-3.5 raw.
- `research` marks rows that contain TabPFN-3.5 (research-only). `no ctx` marks rows containing XGB B'-par, a model without match context. TabPFN v2 here is the D-B' variant (with match context); the extra rows with `TabPFN v2 B'-par` use the pair-only TabPFN variant (D-B'-par, no match context, 10k context, seed 42, all 4 folds; `no ctx`).
- `dBrier` is paired (same bootstrap indices, 2,000 resamples, seed 42) against the raw GRU ensemble (`C3ens raw sym`) and the single raw GRU (`C3 raw sym`); negative = better. `folds` = number of folds (out of 4) with a lower Brier than the reference; `per fold` = the four per-fold dBrier values vs C3ens raw sym.

In [ ]:
from fair_comparison import add_full_grid, acc_ci, acc_score
from sklearn.metrics import roc_auc_score
GG = add_full_grid(B, WORKDIR)
pool = pool_folds(B)   # re-pool so the new columns are included (same rows and order)
yp = pool.Label.values
b_elo, b_elocal = np.mean((pool['p_ELO'] - yp) ** 2), np.mean((pool['p_ELOcal'] - yp) ** 2)
REFS = [('p_C3ens_raw_sym', 'C3ens raw sym'), ('p_C3raw_sym', 'C3 raw sym')]
rows = []
for c, (lab, research, noctx) in GG.items():
    p = pool[c].values
    _, bl, bh = brier_ci(p, yp, boots)
    b = np.mean((p - yp) ** 2)   # point estimate on the full pooled set; CI stays bootstrap percentile
    a, al, ah = acc_ci(p, yp, boots)
    r = {'Blend': lab, 'research': 'yes' if research else '', 'no ctx': 'yes' if noctx else '', 'Brier': b, 'lo': bl, 'hi': bh,
         'vs raw': 100 * (1 - b / b_elo), 'vs cal': 100 * (1 - b / b_elocal),
         'acc': 100 * acc_score(p, yp).mean(), 'alo': 100 * al, 'ahi': 100 * ah, 'AUC': roc_auc_score(yp, p)}
    for ref, nm in REFS:
        d, lo, hi = paired_dbrier(p, pool[ref].values, yp, boots)
        r[nm] = (d, lo, hi, folds_better(B, c, ref))
    r['per fold'] = [x - y for x, y in zip(fold_brier(B, c), fold_brier(B, 'p_C3ens_raw_sym'))]
    rows.append(r)
G = sorted(rows, key=lambda r: r['Brier'])
print(f'ELO raw Brier {b_elo:.5f}, ELO isotonic Brier {b_elocal:.5f}')
print('| Blend | research | no ctx | Brier [95% CI] | % vs ELO raw | % vs ELO cal | Accuracy [95% CI] | AUC | dBrier vs C3ens raw sym [95% CI] | folds | dBrier vs C3 raw sym [95% CI] | folds | per fold dBrier vs C3ens |')
print('|---|---|---|---|---|---|---|---|---|---|---|---|---|')
for r in G:
    x, y = r['C3ens raw sym'], r['C3 raw sym']
    print(f"| {r['Blend']} | {r['research']} | {r['no ctx']} | {r['Brier']:.5f} [{r['lo']:.5f}, {r['hi']:.5f}] | {r['vs raw']:.1f}% | {r['vs cal']:.1f}% | "
          f"{r['acc']:.2f}% [{r['alo']:.2f}, {r['ahi']:.2f}] | {r['AUC']:.4f} | {x[0]:+.5f} [{x[1]:+.5f}, {x[2]:+.5f}] | {x[3]}/4 | "
          f"{y[0]:+.5f} [{y[1]:+.5f}, {y[2]:+.5f}] | {y[3]}/4 | " + ' / '.join(f'{v:+.5f}' for v in r['per fold']) + ' |')

## Figures

Pooled Brier with 95 % CI for the single models, and the full blend grid against the raw GRU ensemble (pooled CI and per-fold dBrier). Saved to `figures/`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
os.makedirs('figures', exist_ok=True)

C_GRU, C_TAB, C_RES, C_ELO = '#1f5fa8', '#d9822b', '#7b3fa0', '#7f7f7f'
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans', 'Arial'], 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#444444',
                     'axes.labelcolor': '#222222', 'xtick.color': '#222222', 'ytick.color': '#222222',
                     'axes.axisbelow': True})

# ---- Figure 1: single models, pooled Brier with 95% CI (raw variant = better variant for every learned model) ----
singles = [('C3 ensemble, raw (5 seeds)', 'p_C3ens_raw', C_GRU), ('C3, raw', 'p_C3raw', C_GRU),
           ('TabPFN-3.5, raw (research-only)', 'p_v35raw', C_RES), ('TabPFN v2, raw', 'p_v2raw', C_TAB),
           ("TabPFN v2 B'-par, raw (no ctx)", 'p_v2par_raw', C_TAB), ("XGBoost B', raw (5 seeds)", 'p_xgb_raw', C_TAB),
           ("XGBoost B'-par, raw (5 seeds, no ctx)", 'p_xgbpar_raw', C_TAB),
           ('ELO, calibrated', 'p_ELOcal', C_ELO), ('ELO, raw', 'p_ELO', C_ELO)]
vals = []
for nm, c, col in singles:
    p = pool[c].values
    _, lo, hi = brier_ci(p, yp, boots)
    vals.append((np.mean((p - yp) ** 2), lo, hi))
fig, ax = plt.subplots(figsize=(10, 5.4), facecolor='white')
for i, ((nm, c, col), (b, lo, hi)) in enumerate(zip(singles, vals)):
    ax.errorbar(b, i, xerr=[[b - lo], [hi - b]], fmt='o', color=col, ecolor=col, capsize=4, capthick=1.6, ms=7, elinewidth=1.8)
    ax.text(hi + 0.0004, i, f'{b:.4f}', va='center', ha='left', fontsize=10, color='#222222')
elo_cal = vals[7][0]
ax.axvline(elo_cal, color='black', ls='--', lw=1)
ax.set_yticks(range(len(singles)))
ax.set_yticklabels([s[0] for s in singles])
ax.invert_yaxis()
ax.set_xlim(min(v[1] for v in vals) - 0.001, max(v[2] for v in vals) + 0.0035)
ax.set_xlabel('Brier score (lower is better)')
ax.grid(axis='x', color='#dddddd', lw=0.7)
ax.tick_params(axis='y', length=0)
ax.set_title('Single models: pooled Brier with 95% bootstrap CI', loc='left', fontsize=12, fontweight='bold')
leg = [Line2D([], [], marker='o', ls='', color=C_GRU, label='GRU family'), Line2D([], [], marker='o', ls='', color=C_TAB, label='Tabular models'),
       Line2D([], [], marker='o', ls='', color=C_RES, label='TabPFN-3.5 (research-only)'),
       Line2D([], [], marker='o', ls='', color=C_ELO, label='ELO'), Line2D([], [], color='black', ls='--', label='ELO, calibrated')]
fig.legend(handles=leg, loc='lower center', ncol=5, frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig('figures/fair_models_brier_ci.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

# ---- Figure 2: blends of the full grid vs C3 ensemble raw (symmetrised) ----
GL = [r for r in G if '+' in r['Blend']]
def lab(r):
    s = r['Blend'].replace(' raw', '').replace(' (5 seeds)', '').replace(' (1 seed)', '').replace(' (5s)', '').replace('TabPFN v2 B\'-par', "TabPFN v2 B'-par*")
    s = s.replace("XGB B'-par", "XGB B'-par*")
    return ' '.join(s.split())
def colr(r):
    return C_RES if r['research'] else (C_TAB if False else C_GRU)
n = len(GL)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 0.42 * n + 2.6), facecolor='white', sharey=True,
                               gridspec_kw={'width_ratios': [1.45, 1], 'wspace': 0.04})
for i, r in enumerate(GL):
    d, lo, hi, _ = r['C3ens raw sym']
    col = C_RES if r['research'] else C_GRU
    ax1.errorbar(d, i, xerr=[[d - lo], [hi - d]], fmt='o' if not r['no ctx'] else 'D', color=col, ecolor=col, capsize=3.5, capthick=1.5, ms=6, elinewidth=1.6)
    ax1.text(0.995, i, f'{d:+.4f}', va='center', ha='right', fontsize=9.5, transform=ax1.get_yaxis_transform())
ax1.axvline(0, color='black', ls='--', lw=1)
ax1.set_yticks(range(n))
ax1.set_yticklabels([lab(r) for r in GL], fontsize=10)
ax1.tick_params(axis='y', length=0)
ax1.invert_yaxis()
ax1.set_xlim(min(r['C3ens raw sym'][1] for r in GL) - 0.0004, max(r['C3ens raw sym'][2] for r in GL) + 0.0020)
ax1.set_xlabel('Paired \u0394Brier vs C3 ensemble raw (negative = better)')
ax1.grid(axis='x', color='#dddddd', lw=0.7)
ax1.set_title('A  Pooled \u0394Brier, 95% CI', loc='left', fontsize=12, fontweight='bold')
M = np.array([r['per fold'] for r in GL]) * 1000
lim = np.abs(M).max()
im = ax2.imshow(M, aspect='auto', cmap='RdYlGn_r', norm=TwoSlopeNorm(0, -lim, lim))
for i in range(n):
    for j in range(M.shape[1]):
        ax2.text(j, i, f'{M[i, j]:+.1f}', ha='center', va='center', fontsize=9.5, color='white' if abs(M[i, j]) > 3.3 else 'black')
ax2.set_xticks(range(M.shape[1]))
ax2.set_xticklabels([f'Fold {k}' for k in folds])
ax2.xaxis.set_ticks_position('bottom')
ax2.tick_params(length=0)
for s in ax2.spines.values():
    s.set_visible(False)
ax2.set_title('B  \u0394Brier by fold (\u00d710\u00b3)', loc='left', fontsize=12, fontweight='bold')
cb = fig.colorbar(im, ax=ax2, fraction=0.05, pad=0.03)
cb.set_label('\u0394Brier \u00d710\u00b3 (green = better, red = worse)')
cb.outline.set_visible(False)
leg = [Line2D([], [], marker='o', ls='', color=C_GRU, label='Blend'), Line2D([], [], marker='o', ls='', color=C_RES, label='Contains TabPFN-3.5 (research-only)'),
       Line2D([], [], marker='D', ls='', color=C_GRU, label="* = no match context ('no ctx'), diamond marker")]
fig.legend(handles=leg, loc='lower center', ncol=3, frameon=False, fontsize=10, bbox_to_anchor=(0.45, -0.01))
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig('figures/fair_blends_by_fold.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

## Conclusion

**Ranking and calibration.**
- **The order of the models is the same in all four views** (raw, calibrated, and both symmetrised): C3ens, C3, TabPFN-3.5, TabPFN v2, XGBoost B', XGBoost B'-par, ELO. Calibration changes the Brier of the learned models by +0.0004 to +0.0009 (it makes them slightly worse; the paired CI is above 0 for all of them, the calibrated version wins in 0-1 of 4 folds), while it improves ELO by -0.0045 (CI below 0, 4 of 4 folds). Raw outputs are therefore the fairer comparison for the learned models, and the calibrated ELO is the stricter baseline.
- **The percentage gain over ELO drops by about 2.3-2.6 points when the reference is the calibrated ELO:** C3ens raw 10.1 % -> 7.8 %, C3 raw 9.6 % -> 7.3 %, TabPFN v2 raw 8.5 % -> 6.2 %, XGBoost B' (5 seeds, raw) 8.0 % -> 5.6 %, and the best allowed blend (C3 ensemble + TabPFN v2) 10.9 % vs raw ELO -> 8.6 % vs calibrated ELO (the research-only TabPFN-3.5 blend 11.3 % -> 9.0 %). The gap to ELO stays statistically clear: C3 raw vs ELO isotonic dBrier -0.0125, XGBoost B' vs ELO isotonic -0.0097, both CIs far below 0, 4 of 4 folds. Accuracy is almost unchanged (the isotonic ELO is 0.2 pp less accurate than raw ELO); the models are 0.7-1.9 pp above raw ELO.
- **Seed parity matters for XGBoost.** Seed 42 was the worst of the five XGBoost B' seeds; the 5-seed ensemble lowers Brier by 0.00075 (raw) / 0.00135 (calibrated), CI below 0. With 5 seeds the gap to TabPFN v2 (raw dBrier +0.0009, CI [-0.0001, +0.0019], 1 of 4 folds) is no longer statistically clear, which it was against the single-seed XGBoost in notebook 16. TabPFN-3.5 stays ahead of TabPFN v2 (-0.0014, CI below 0), and C3ens stays ahead of C3 (-0.0009, 4 of 4 folds) but not clearly ahead of TabPFN-3.5 (CI includes 0).

**Blends.**
- **Blending still helps, with slightly smaller margins against the stronger reference.** 0.5 C3ens raw + 0.5 XGBoost B' beats C3ens raw by -0.0012 (CI [-0.0020, -0.0004], 3 of 4 folds) and C3 raw by -0.0021 (4 of 4 folds); the pair-only B'-par blend by -0.0009 (CI [-0.0018, -0.0001], 3 of 4 folds) and -0.0018. Whether the XGBoost part is raw or calibrated, and whether the blend is symmetrised, changes the Brier by at most 0.0001, far inside the CIs. The B'-par blend with calibrated XGBoost has a CI that just touches 0 against C3ens raw (and the all-calibrated B'-par blend includes 0), and the symmetrised raw B'-par blend beats the symmetrised GRU ensemble in only 2 of 4 folds.
- **Symmetrisation** changes pooled Brier by at most about 0.0001 for the 5-seed ensembles and blends, and by up to 0.0003 for the calibrated single GRU.

**Limitations.**
- Weights are fixed 0.5 / 0.5, not tuned. TabPFN is one run per fold (internal ensemble of 8) and is not seed-matched.
- "Calibrated" means each model's own calibrator on the common calibration slice: isotonic for ELO, XGBoost and TabPFN, temperature scaling or isotonic (whichever scores better on that slice) for the GRU. The Platt and isotonic ELO differ by 0.0004 (CI includes 0).
- Hyper-parameters were not re-tuned for this comparison. The XGBoost values come from a 3-point grid with a time-series CV inside the train part (2017-05 to 2025-01) of an earlier, smaller export (notebook 03), and the GRU values were set once (notebook 05) without a search; the matches before 2025-01 are also test matches of folds 1-2 and most of fold 3 here, so those folds are not tuning-free, whereas fold 4 (2025-06 to 2026-11) lies after the tuning window. Feature sets (B', C3 head) were likewise developed and first evaluated on the earlier export's single split and walk-forward folds.
- Test matches are shared across models; only paired CIs are used to compare.